In [42]:
# %pip install langchain langchain-openai langchain-community langchain-text-splitters

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate  # Changed to langchain_core
from langchain_classic.chains.combine_documents import create_stuff_documents_chain  # Changed to classic
from langchain_classic.chains import create_retrieval_chain 

In [44]:
# # Load environment variables
# load_dotenv()

# # Check if API key is set
# if not os.getenv("OPENAI_API_KEY"):
#     raise ValueError("OPENAI_API_KEY not found. Please set it in .env file")

# Load environment variables
load_dotenv()

def validate_api_key():
    """Check if OpenAI API key is properly set"""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not found!")
        print("Please set it in one of these ways:")
        print("1. Create a .env file with: OPENAI_API_KEY=your-key-here")
        print("2. Set environment variable: export OPENAI_API_KEY=your-key-here")
        print("3. Set directly in code: os.environ['OPENAI_API_KEY'] = 'your-key'")
        return False
    
    if not api_key.startswith("sk-"):
        print("ERROR: Invalid API key format. Key should start with 'sk-'")
        return False
    
    if len(api_key) < 20:
        print("ERROR: API key seems too short. Please check if it's complete.")
        return False
    
    print(f"API key found (starts with: {api_key[:10]}...)")
    return True

In [45]:
# 1. Load docs
def load_docs():
    """Load documents from docs directory"""
    if not os.path.exists("docs"):
        os.makedirs("docs")
        print("Created 'docs' directory. Please add your documents there.")
        return []
    
    # Try to load all supported files
    all_docs = []
    
    # Load text files
    text_files = [f for f in os.listdir("docs") if f.endswith(('.txt', '.md', '.text'))]
    for file in text_files:
        try:
            loader = TextLoader(os.path.join("docs", file), encoding="utf-8")
            all_docs.extend(loader.load())
            print(f"Loaded: {file}")
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    # Load PDF files
    pdf_files = [f for f in os.listdir("docs") if f.endswith('.pdf')]
    for file in pdf_files:
        try:
            loader = PyPDFLoader(os.path.join("docs", file))
            all_docs.extend(loader.load())
            print(f"Loaded: {file}")
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    if not all_docs:
        print("No documents found in 'docs' directory!")
        print("Please add .txt or .pdf files to the 'docs' folder.")
    
    return all_docs

In [46]:
# 2. Create vector store
def build_vectorstore(docs):
    """Create vector store from documents"""
    if not docs:
        return None
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
    chunks = splitter.split_documents(docs)
    print(f"Split into {len(chunks)} chunks")
    
    embeddings = OpenAIEmbeddings()
    vectordb = Chroma.from_documents(
        chunks, 
        embedding=embeddings, 
        collection_name="demo-rag",
        persist_directory="./chroma_db"
    )
    return vectordb

In [47]:
# 3. Build RAG chain
def build_rag_chain(vectordb):
    """Build RAG chain"""
    if not vectordb:
        return None
    
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

    prompt = ChatPromptTemplate.from_template(
        "You are a helpful assistant. Use the following context to answer the question.\n"
        "If the answer is not in the context, say you don't know.\n\n"
        "Context:\n{context}\n\nQuestion: {input}"
    )

    qa_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, qa_chain)
    return rag_chain

In [48]:
# 4. Main function
def get_answer(question: str) -> str:
    """Get answer from RAG system"""
    if not validate_api_key():
        return "Please set up your OpenAI API key first."
    
    if not question:
        return "Please provide a question."
    
    print(f"\nProcessing: {question}")
    
    docs = load_docs()
    if not docs:
        return "No documents to process. Please add files to the 'docs' directory."
    
    vectordb = build_vectorstore(docs)
    if not vectordb:
        return "Failed to create vector store."
    
    rag_chain = build_rag_chain(vectordb)
    if not rag_chain:
        return "Failed to build RAG chain."
    
    try:
        result = rag_chain.invoke({"input": question})
        return result["answer"]
    except Exception as e:
        return f"Error: {str(e)}"

In [49]:
# #  Interactive mode
# def interactive_mode():
#     """Run in interactive mode"""
#     print("\n=== RAG Assistant ===")
#     print("Type 'quit' or 'exit' to stop\n")
    
#     # Preload documents for faster response
#     print("Loading documents...")
#     docs = load_docs()
#     if not docs:
#         print("Please add documents to the 'docs' directory and restart.")
#         return
    
#     vectordb = build_vectorstore(docs)
#     if not vectordb:
#         return
    
#     rag_chain = build_rag_chain(vectordb)
#     if not rag_chain:
#         return
    
    # print(f"Loaded {len(docs)} document(s). Ready for questions!\n")
    
    # while True:
    #     question = input("Ask a question: ").strip()
    #     if question.lower() in ['quit', 'exit', 'q']:
    #         print("Goodbye!")
    #         break
    #     if not question:
    #         continue
        
    #     try:
    #         result = rag_chain.invoke({"input": question})
    #         print(f"\nAnswer: {result['answer']}\n")
    #     except Exception as e:
    #         print(f"\nError: {str(e)}\n")

In [ ]:
# Set the key
os.environ["OPENAI_API_KEY"] = 'your-key-here'

In [56]:
print("=== RAG System ===")
q = input("\nAsk a question: ")
answer = get_answer(q)
print(f"\nAnswer: {answer}")

=== RAG System ===
API key found (starts with: sk-proj-Ti...)

Processing: what is paracetomol
Loaded: guide.pdf
Split into 4479 chunks


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}